In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
BASE_DIR = Path('../../')
FIG_DIR  = BASE_DIR / 'paper/output/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

matplotlib.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12,
                            'axes.spines.top': False, 'axes.spines.right': False})

---
## Tabla de Resultados

Completar con los valores obtenidos al ejecutar los notebooks 06–13.

- `Train_AUC`: AUC sobre el conjunto de entrenamiento completo
- `CV_AUC`: Promedio de validación cruzada (5-fold, excepto NN que usa 20% val)
- `CV_std`: Desviación estándar del CV AUC (None para modelos sin std)
- `Public_LB`: AUC en el Public Leaderboard de Kaggle (~30% test)
- `Private_LB`: AUC en el Private Leaderboard (~70% test)
- `tipo`: 'Econométrico' o 'ML'

In [ ]:
# ============================================================
# COMPLETAR CON LOS RESULTADOS DE CADA NOTEBOOK
# ============================================================
results = {
    '06_Logit_MLE': {
        'Train_AUC': 0.7498, 'CV_AUC': 0.7498, 'CV_std': 0.0053,
        'Public_LB': 0.74978, 'Private_LB': 0.73499, 'tipo': 'Econométrico'
    },
    '07_Logit_L1L2': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'Econométrico'
    },
    '08a_Spline_Logit': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'Econométrico'
    },
    '08b_GAM': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'Econométrico'
    },
    '09_Random_Forest': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'ML'
    },
    '10_XGBoost': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'ML'
    },
    '11_LightGBM': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'ML'
    },
    '12_CatBoost': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'ML'
    },
    '13_Neural_Net': {
        'Train_AUC': None, 'CV_AUC': None, 'CV_std': None,
        'Public_LB': None, 'Private_LB': None, 'tipo': 'ML'
    },
}
# ============================================================

df_res = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Modelo'})
df_res = df_res.astype({'Train_AUC': float, 'CV_AUC': float})

# Columna de ganancia sobre baseline logit
baseline_cv = df_res.loc[df_res['Modelo'] == '06_Logit_MLE', 'CV_AUC'].values[0]
df_res['Gain_vs_Logit'] = (df_res['CV_AUC'] - baseline_cv).round(4)

display(df_res[['Modelo','tipo','Train_AUC','CV_AUC','CV_std','Public_LB','Private_LB','Gain_vs_Logit']])

---
## Visualizaciones

In [ ]:
# Filtrar solo filas con CV_AUC disponible
df_plot = df_res.dropna(subset=['CV_AUC', 'Public_LB']).copy()

COLOR_MAP = {'Econométrico': '#3498db', 'ML': '#e74c3c'}
colors    = [COLOR_MAP[t] for t in df_plot['tipo']]

modelos   = df_plot['Modelo'].tolist()
n         = len(modelos)
x         = np.arange(n)
w         = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - w/2, df_plot['CV_AUC'],    width=w, label='CV AUC (train)',  color=colors, alpha=0.85)
bars2 = ax.bar(x + w/2, df_plot['Public_LB'], width=w, label='Public LB (test)',color=colors, alpha=0.5, hatch='//')

# Etiquetas de valor
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(modelos, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('AUC'); ax.set_title('Comparación de Modelos — AUC')
ax.set_ylim(0.70, min(1.0, df_plot[['CV_AUC','Public_LB']].max().max() + 0.02))

# Leyenda tipo
patch_eco = mpatches.Patch(color='#3498db', label='Econométrico')
patch_ml  = mpatches.Patch(color='#e74c3c', label='ML')
bar_cv    = mpatches.Patch(color='gray', alpha=0.85, label='CV AUC')
bar_lb    = mpatches.Patch(color='gray', alpha=0.5,  hatch='//', label='Public LB')
ax.legend(handles=[patch_eco, patch_ml, bar_cv, bar_lb], loc='lower right')

plt.tight_layout()
fig.savefig(FIG_DIR / '14_auc_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figura guardada: 14_auc_comparison.pdf')

In [ ]:
# Scatter Train AUC vs CV AUC — análisis de overfitting
df_ov = df_res.dropna(subset=['Train_AUC', 'CV_AUC']).copy()

fig, ax = plt.subplots(figsize=(9, 7))
for _, row in df_ov.iterrows():
    c = COLOR_MAP[row['tipo']]
    ax.scatter(row['CV_AUC'], row['Train_AUC'], color=c, s=100, zorder=3)
    ax.annotate(row['Modelo'], (row['CV_AUC'], row['Train_AUC']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

# Línea diagonal: Train AUC = CV AUC (sin overfitting)
lim_min = df_ov[['Train_AUC','CV_AUC']].min().min() - 0.005
lim_max = df_ov[['Train_AUC','CV_AUC']].max().max() + 0.005
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1, alpha=0.4, label='Sin overfitting')

ax.set_xlabel('CV AUC'); ax.set_ylabel('Train AUC')
ax.set_title('Análisis de Overfitting — Train AUC vs CV AUC')
ax.legend(handles=[mpatches.Patch(color='#3498db', label='Econométrico'),
                   mpatches.Patch(color='#e74c3c', label='ML'),
                   plt.Line2D([0],[0],color='k',ls='--',lw=1,alpha=0.4,label='Sin overfitting')])
plt.tight_layout()
fig.savefig(FIG_DIR / '14_overfitting_analysis.pdf', bbox_inches='tight')
plt.show()
print('Figura guardada: 14_overfitting_analysis.pdf')

In [ ]:
# Ganancia sobre baseline logit
df_gain = df_res.dropna(subset=['CV_AUC']).copy()
df_gain = df_gain.sort_values('Gain_vs_Logit', ascending=True)

gain_colors = ['#2ecc71' if g >= 0 else '#e74c3c' for g in df_gain['Gain_vs_Logit']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_gain['Modelo'], df_gain['Gain_vs_Logit'], color=gain_colors, alpha=0.8)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Diferencia de CV AUC vs Logit MLE')
ax.set_title('Ganancia sobre Baseline (Logit MLE)')
plt.tight_layout()
fig.savefig(FIG_DIR / '14_gain_over_baseline.pdf', bbox_inches='tight')
plt.show()
print('Figura guardada: 14_gain_over_baseline.pdf')

---
## Análisis por Grupo: Econométrico vs ML

In [ ]:
df_valid = df_res.dropna(subset=['CV_AUC'])

grupo = df_valid.groupby('tipo')[['CV_AUC', 'Public_LB', 'Gain_vs_Logit']].agg(['mean','std'])
print('Promedio por grupo:')
display(grupo.round(4))

print('\nRanking por CV AUC:')
ranking = df_valid[['Modelo','tipo','CV_AUC','Public_LB','Gain_vs_Logit']].sort_values('CV_AUC', ascending=False)
display(ranking.reset_index(drop=True))

---
## Mejor Modelo

In [ ]:
df_valid = df_res.dropna(subset=['CV_AUC'])

best_cv = df_valid.loc[df_valid['CV_AUC'].idxmax()]
best_lb = df_valid.dropna(subset=['Public_LB'])
best_lb = best_lb.loc[best_lb['Public_LB'].idxmax()] if len(best_lb) > 0 else None

print('=' * 65)
print('MEJOR MODELO — RESUMEN')
print('=' * 65)
print(f'Por CV AUC    : {best_cv["Modelo"]}  (CV AUC = {best_cv["CV_AUC"]:.4f}, tipo={best_cv["tipo"]})')
if best_lb is not None:
    print(f'Por Public LB : {best_lb["Modelo"]}  (Public LB = {best_lb["Public_LB"]:.4f}, tipo={best_lb["tipo"]})')

eco_mean = df_valid[df_valid['tipo'] == 'Econométrico']['CV_AUC'].mean()
ml_mean  = df_valid[df_valid['tipo'] == 'ML']['CV_AUC'].mean()
print(f'\nPromedio CV AUC Econométrico : {eco_mean:.4f}')
print(f'Promedio CV AUC ML           : {ml_mean:.4f}')
print(f'Diferencia ML - Econométrico : {ml_mean - eco_mean:+.4f}')
print('=' * 65)